In [ ]:
%load_ext autoreload
%autoreload 2

from model.model import LavaTubeFinder
from model.training_utils import *
import torch
from model.hirise_dataset import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Using cuda" if torch.cuda.is_available() else "Using cpu")


In [ ]:
image_data = pd.read_json("data/optical/final_data/image_dataset.json")
image_data

In [ ]:
from torchvision.transforms import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.CenterCrop(1000)
])

image_dataset = Hirise_Dataset(image_data, root_dir=".", transform=transform)
train_dataloader, val_dataloader = create_dataloaders(image_dataset, batch_size=4)

In [4]:
for batch_idx, (images, thermals, labels) in enumerate(train_dataloader):
    print(f"--- Batch {batch_idx + 1} ---")
    print(f"Images shape:   {images.shape}   | dtype: {images.dtype} | device: {images.device}")
    print(f"Thermals shape: {thermals.shape} | dtype: {thermals.dtype} | device: {thermals.device}")
    print(f"Labels shape:   {labels.shape}   | values: {labels.tolist()}")

    if batch_idx == 1:  # Stops after 2 batches (index 0 and 1)
        break

--- Batch 1 ---
Images shape:   torch.Size([4, 1, 1000, 1000])   | dtype: torch.float32 | device: cpu
Thermals shape: torch.Size([4, 3, 1, 32, 32]) | dtype: torch.float32 | device: cpu
Labels shape:   torch.Size([4])   | values: [3, 0, 3, 3]
--- Batch 2 ---
Images shape:   torch.Size([4, 1, 1000, 1000])   | dtype: torch.float32 | device: cpu
Thermals shape: torch.Size([4, 3, 1, 32, 32]) | dtype: torch.float32 | device: cpu
Labels shape:   torch.Size([4])   | values: [3, 1, 3, 0]


In [5]:
SINGLE_MODEL_TRAIN = False

if SINGLE_MODEL_TRAIN:
    lava_model = LavaTubeFinder(n_classes=4, modality="both")  # "optical" | "thermal" | "both"

    history = train_model(
        model=lava_model,
        train_loader=train_dataloader,
        val_loader=val_dataloader,
        criterion=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam(lava_model.parameters(), lr=0.0001),
        num_epochs=15,
        scheduler=None,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    )

In [6]:
if SINGLE_MODEL_TRAIN:
    epochs_range = range(1, len(history["train_loss"]) + 1)

    metrics = ["loss", "acc", "precision", "recall", "f1"]
    titles = ["Loss", "Accuracy", "Precision", "Recall", "F1 Score"]

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    axs = axs.flatten()

    for ax, metric, title in zip(axs, metrics, titles):
        ax.plot(epochs_range, history[f"train_{metric}"], label="Train", marker="o")
        ax.plot(epochs_range, history[f"val_{metric}"], label="Validation", marker="o")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.legend()
        ax.grid(alpha=0.3)

    axs[-1].axis("off")  # unused subplot (5 metrics in a 2x3 grid)

    plt.tight_layout()
    plt.show()

In [7]:
if SINGLE_MODEL_TRAIN:
    class_names = {0: "Vertical Pit/Skylight", 1: "Shallow Pit Chain", 2: "Sloped Pit", 3: "Negative (same tile)"}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    lava_model.eval()

    sample_idx = np.random.randint(len(val_dataloader.dataset))
    image, thermal, true_label = val_dataloader.dataset[sample_idx]

    with torch.no_grad():
        logits = lava_model(
            image.unsqueeze(0).to(device),
            thermal.unsqueeze(0).to(device),
        )
        predicted_label = torch.argmax(logits, dim=1).item()

    plt.imshow(image.squeeze(0).cpu(), cmap="gray")
    plt.title(
        f"Predicted: {class_names[predicted_label]} | True: {class_names[true_label]}"
    )
    plt.axis("off")
    plt.show()

## Modality ablation

Train the same architecture on the optical crop alone, the thermal sequence
alone, and both, to see what each stream actually contributes.

Two things to keep in mind when reading the result:

* The thermal sequence is still the synthetic `torch.randn` placeholder until
  real THEMIS windows are wired in, so **"thermal" should score near chance
  (25%) and "both" should not beat "optical"**. Anything else means the model
  is finding signal in noise -- a bug worth chasing.
* Splits are group-aware, so a landform cannot appear in both train and
  validation.

For reference, a classifier using only low-level image statistics -- no
morphology at all -- reaches roughly 70% on this 4-class problem, because the
two source datasets were collected with different pipelines. Treat that as the
bar to clear, not 25%.

In [ ]:
MODALITY_EPOCHS = 5

histories = {}

for modality in ("optical", "thermal", "both"):
    print(f"\n\n{'#' * 65}")

    # Skip decoding the large HiRISE crops when they will not be used.
    dataset = Hirise_Dataset(
        image_data,
        root_dir=".",
        transform=transform,
        load_optical=(modality != "thermal"),
    )
    train_dl, val_dl = create_dataloaders(dataset, batch_size=4)

    model = LavaTubeFinder(n_classes=4, modality=modality)
    histories[modality] = train_model(
        model=model,
        train_loader=train_dl,
        val_loader=val_dl,
        criterion=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam(model.parameters(), lr=0.0001),
        scheduler=None,
        num_epochs=MODALITY_EPOCHS,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        save_path=f"best_model_{modality}.pt",
    )

In [ ]:
# Metrics at each run's best-F1 epoch (F1 rather than accuracy, since the
# classes are imbalanced).
metrics = ["val_acc", "val_precision", "val_recall", "val_f1"]

summary = pd.DataFrame({
    modality: {
        m: hist[m][int(np.argmax(hist["val_f1"]))] for m in metrics
    }
    for modality, hist in histories.items()
}).T
summary.columns = [c.replace("val_", "") for c in summary.columns]

print("Validation metrics at best-F1 epoch:")
display((summary * 100).round(2))

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

summary.plot.bar(ax=axs[0], rot=0)
axs[0].axhline(0.25, ls="--", c="grey", lw=1, label="chance (4 classes)")
axs[0].set_title("Validation metrics by modality")
axs[0].set_ylabel("score")
axs[0].legend(fontsize=8)
axs[0].grid(alpha=0.3, axis="y")

for modality, hist in histories.items():
    axs[1].plot(range(1, len(hist["val_f1"]) + 1), hist["val_f1"],
                marker="o", label=modality)
axs[1].axhline(0.25, ls="--", c="grey", lw=1)
axs[1].set_title("Validation F1 per epoch")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("F1 (macro)")
axs[1].legend()
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Cross-validated modality ablation

A single 80/20 split is too noisy to trust here. The matched dataset holds 1560
crops but only **229 independent groups** (a group covers one landform, all its
resolution replicas, and both halves of its crop pair), so one hold-out leaves
about 45 validation groups.

`cross_validate` runs stratified, group-aware k-fold: no group straddles a fold,
class balance is held across folds, and every fold trains a fresh model.

**How to read the result.** On the matched crops a classifier using only image
statistics -- no morphology -- reaches:

| features | 4-class | negative vs rest |
|---|---|---|
| acquisition (resolution, size, extent) | 47.6% | 50.0% |
| intensity | 55.1% | 67.1% |
| all statistics | **58.0%** | **69.4%** |
| majority class | 50.0% | 50.0% |

Acquisition features sit at chance, which is the point of the matched pairs.
The intensity number is *not* a shortcut -- a pit casts a real shadow -- so
**58.0% is the bar to clear on 4-class**, not 25%.

The thermal sequence is still the `torch.randn` placeholder, so "thermal" should
land near chance and "both" should not beat "optical". Anything else is a bug.

In [ ]:
from functools import partial

CV_FOLDS = 5
CV_EPOCHS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cv_results = {}

for modality in ("optical", "thermal", "both"):
    print(f"{'#' * 65}")

    dataset = Hirise_Dataset(
        image_data,
        root_dir=".",
        transform=transform,
        load_optical=(modality != "thermal"),
    )

    per_fold, _ = cross_validate(
        image_dataset=dataset,
        model_factory=partial(LavaTubeFinder, n_classes=4, modality=modality),
        criterion=nn.CrossEntropyLoss(),
        optimizer_factory=lambda m: torch.optim.Adam(m.parameters(), lr=0.0001),
        num_epochs=CV_EPOCHS,
        device=device,
        n_splits=CV_FOLDS,
        batch_size=4,
    )
    cv_results[modality] = per_fold

In [ ]:
metrics = ["val_acc", "val_precision", "val_recall", "val_f1"]

summary = pd.DataFrame({
    modality: {
        f"{m.replace('val_', '')}": f"{per_fold[m].mean()*100:.1f} ± {per_fold[m].std()*100:.1f}"
        for m in metrics
    }
    for modality, per_fold in cv_results.items()
}).T

print(f"Mean ± std across {CV_FOLDS} folds:")
display(summary)

# The no-morphology baseline the model has to beat (see the markdown above).
STATS_BASELINE_4CLASS = 0.580

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

means = pd.DataFrame({m: p[metrics].mean() for m, p in cv_results.items()}).T
errs = pd.DataFrame({m: p[metrics].std() for m, p in cv_results.items()}).T
means.plot.bar(yerr=errs, ax=axs[0], rot=0, capsize=3)
axs[0].axhline(STATS_BASELINE_4CLASS, ls="--", c="crimson", lw=1.5,
               label=f"image-statistics baseline ({STATS_BASELINE_4CLASS:.0%})")
axs[0].axhline(0.50, ls=":", c="grey", lw=1, label="majority class (50%)")
axs[0].set_title(f"Validation metrics, mean ± std over {CV_FOLDS} folds")
axs[0].legend(fontsize=8)
axs[0].grid(alpha=0.3, axis="y")

for modality, per_fold in cv_results.items():
    axs[1].scatter(per_fold.index, per_fold["val_f1"], label=modality, s=45)
axs[1].axhline(STATS_BASELINE_4CLASS, ls="--", c="crimson", lw=1.5)
axs[1].set_title("Per-fold F1 (spread shows how noisy a single split would be)")
axs[1].set_xlabel("fold")
axs[1].set_ylabel("F1 (macro)")
axs[1].legend()
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()